# YOLO26 Object Detection Training

This notebook documents the complete YOLO26 Nano workflow: environment setup, dataset preparation, two training experiments, held-out test evaluation, evaluation artifact review, and a single-image TP/FP/FN example.

The notebook is designed for **Google Colab with a GPU runtime**. It uses the project dataset archive stored in Google Drive.

## 1. Environment Setup


In [ ]:
%pip install -q ultralytics==8.4.211


In [ ]:
import torch
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)


## 2. Google Drive and Dataset Preparation

Mount Google Drive and extract the dataset archive. The archive should be stored at:

`MyDrive/YOLO26_Project/YOLO26_Dataset.zip`

The archive should contain `images/`, `labels/`, and `data.yaml`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil
import yaml

PROJECT_ROOT = Path("/content/drive/MyDrive/YOLO26_Project")
ZIP_PATH = PROJECT_ROOT / "YOLO26_Dataset.zip"
DATASET_DIR = Path("/content/YOLO26_Dataset")

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {ZIP_PATH}")

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

DATASET_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH, "r") as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue

        relative_path = Path(member.filename.replace("\\\\", "/"))
        destination = (DATASET_DIR / relative_path).resolve()

        if not destination.is_relative_to(DATASET_DIR.resolve()):
            raise ValueError(f"Unsafe archive path: {member.filename}")

        destination.parent.mkdir(parents=True, exist_ok=True)

        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)

print("Dataset extraction completed.")
print("Dataset directory:", DATASET_DIR)


In [ ]:
# Verify that every image has exactly one matching label file.
image_extensions = {".jpg", ".jpeg", ".png"}

for split in ["train", "val", "test"]:
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"Missing image or label directory for {split}.")

    images = [
        file for file in image_dir.iterdir()
        if file.is_file() and file.suffix.lower() in image_extensions
    ]
    labels = list(label_dir.glob("*.txt"))

    image_stems = [file.stem for file in images]
    label_stems = [file.stem for file in labels]

    if len(image_stems) != len(set(image_stems)):
        raise ValueError(f"Duplicate image filenames in {split} split.")

    if len(label_stems) != len(set(label_stems)):
        raise ValueError(f"Duplicate label filenames in {split} split.")

    missing_labels = set(image_stems) - set(label_stems)
    extra_labels = set(label_stems) - set(image_stems)

    print(f"{split.upper()}: Images = {len(images)}, Labels = {len(labels)}")

    if missing_labels or extra_labels:
        raise ValueError(
            f"Image-label mismatch in {split}: "
            f"{len(missing_labels)} missing labels, "
            f"{len(extra_labels)} extra labels."
        )

YAML_PATH = DATASET_DIR / "data.yaml"

if not YAML_PATH.is_file():
    raise FileNotFoundError(f"Dataset YAML not found: {YAML_PATH}")


In [ ]:
# Create a Colab-specific YAML file without changing the original data.yaml.
with YAML_PATH.open("r", encoding="utf-8-sig") as file:
    dataset_config = yaml.safe_load(file)

if not isinstance(dataset_config, dict):
    raise ValueError("data.yaml must contain a YAML mapping.")

required_keys = {"train", "val", "test", "nc", "names"}
missing_keys = required_keys - set(dataset_config)

if missing_keys:
    raise ValueError(f"data.yaml is missing keys: {sorted(missing_keys)}")

dataset_config["path"] = str(DATASET_DIR)

COLAB_YAML_PATH = DATASET_DIR / "data_colab.yaml"

with COLAB_YAML_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, sort_keys=False)

DATA_YAML = str(COLAB_YAML_PATH)

for split in ["train", "val", "test"]:
    split_path = DATASET_DIR / dataset_config[split]
    if not split_path.exists():
        raise FileNotFoundError(f"Missing {split} directory: {split_path}")

print("Colab dataset configuration:")
print(yaml.safe_dump(dataset_config, sort_keys=False))
print("Configuration saved to:", DATA_YAML)


## 3. Training Configuration

Both experiments use YOLO26 Nano and the same dataset, image size, batch size, optimizer, and random seed. Experiment 2 uses a lower initial learning rate and a longer early-stopping patience.


In [ ]:
from ultralytics import YOLO

RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_ROOT)
print("Training results directory:", RUNS_DIR)
print("Dataset YAML:", DATA_YAML)


## 4. Experiment 1 — Baseline YOLO26n

Initial learning rate: **0.01**.


In [ ]:
baseline_model = YOLO("yolo26n.pt")

baseline_results = baseline_model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=15,
    seed=42,
    deterministic=True,
    pretrained=True,
    amp=True,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name="baseline_yolo26n",
    exist_ok=False,
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)


### Baseline Training Results

Read the recorded training history and identify the epoch with the highest validation mAP50-95.


In [ ]:
import pandas as pd
from IPython.display import display

baseline_dir = RUNS_DIR / "baseline_yolo26n"
results_file = baseline_dir / "results.csv"

if not results_file.is_file():
    raise FileNotFoundError(f"Training history not found: {results_file}")

df = pd.read_csv(results_file)
df.columns = df.columns.str.strip()

print("Total epochs recorded:", len(df))

metric_cols = [column for column in df.columns if column.startswith("metrics/")]
loss_cols = [
    column for column in df.columns
    if column.startswith("train/") or column.startswith("val/")
]

selected_cols = ["epoch"] + loss_cols + metric_cols
display(df[selected_cols].round(4))

map_col = "metrics/mAP50-95(B)"

if map_col in df.columns and not df[map_col].dropna().empty:
    best_row = df.loc[df[map_col].idxmax()]
    print("\nBest validation mAP50-95:")
    print("Epoch:", int(best_row["epoch"]))
    print("mAP50-95:", round(float(best_row[map_col]), 4))
else:
    print("The mAP50-95 column was not found in the training history.")


## 5. Experiment 2 — Reduced Learning Rate

The second experiment uses an initial learning rate of **0.001**, retaining the other main training settings.


In [ ]:
experiment2_model = YOLO("yolo26n.pt")

experiment2_results = experiment2_model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    optimizer="SGD",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    seed=42,
    deterministic=True,
    pretrained=True,
    amp=True,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name="experiment2_yolo26n_lr001",
    exist_ok=False,
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)


## 6. Final Test Evaluation

Load the best checkpoint from Experiment 2. The final held-out test evaluation has already been completed and is not repeated by default.

Set `RUN_FINAL_TEST_EVALUATION = True` only when you deliberately want to create a new test evaluation.


In [ ]:
BEST_MODEL_PATH = (
    RUNS_DIR
    / "experiment2_yolo26n_lr001"
    / "weights"
    / "best.pt"
)

if not BEST_MODEL_PATH.is_file():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_MODEL_PATH}")

best_model = YOLO(str(BEST_MODEL_PATH))

print("Selected checkpoint:", BEST_MODEL_PATH)
print("Checkpoint size (MB):", round(BEST_MODEL_PATH.stat().st_size / (1024 * 1024), 2))


In [ ]:
RUN_FINAL_TEST_EVALUATION = False

EVALUATION_ROOT = PROJECT_ROOT / "final_test_evaluation"
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)

# Default location of the previously completed evaluation.
EVALUATION_DIR = EVALUATION_ROOT / "experiment2_test_verified_1"

if RUN_FINAL_TEST_EVALUATION:
    evaluation_name = "experiment2_test_verified"
    counter = 1

    while (EVALUATION_ROOT / evaluation_name).exists():
        evaluation_name = f"experiment2_test_verified_{counter}"
        counter += 1

    EVALUATION_DIR = EVALUATION_ROOT / evaluation_name

    test_results = best_model.val(
        data=DATA_YAML,
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        plots=True,
        project=str(EVALUATION_ROOT),
        name=evaluation_name,
        exist_ok=False,
        verbose=True
    )

    print("\nFINAL TEST RESULTS")
    print("==================")
    print(f"Precision:  {test_results.box.mp:.5f}")
    print(f"Recall:     {test_results.box.mr:.5f}")
    print(f"mAP50:      {test_results.box.map50:.5f}")
    print(f"mAP50-95:   {test_results.box.map:.5f}")
    print("Evaluation artifacts:", EVALUATION_DIR)
else:
    print("Using the previously completed test evaluation.")
    print("Evaluation artifacts:", EVALUATION_DIR)


## 7. Evaluation Artifacts

Review the files produced by the held-out test evaluation. This cell uses the evaluation directory selected in the previous cell.


In [ ]:
if not EVALUATION_DIR.is_dir():
    raise FileNotFoundError(
        f"Evaluation directory not found: {EVALUATION_DIR}. "
        "Check the saved evaluation folder or enable a new test evaluation."
    )

print("VERIFIED FINAL EVALUATION ARTIFACTS")
print("===================================")

evaluation_files = sorted(
    file for file in EVALUATION_DIR.rglob("*")
    if file.is_file()
)

for file in evaluation_files:
    size_kb = file.stat().st_size / 1024
    print(f"{file.name:35} {size_kb:.1f} KB")

print("\nTotal files:", len(evaluation_files))


## 8. Single-Image TP, FP and FN Analysis

This section evaluates the worked-example test image `000009.jpg`. Predictions are matched to ground-truth boxes using class agreement and an IoU threshold of 0.50. Each ground-truth object can be matched only once.

These are image-level results, not the overall test-set metrics.


In [ ]:
from PIL import Image
import math

IMAGE_DIR = DATASET_DIR / "images" / "test"
LABEL_DIR = DATASET_DIR / "labels" / "test"

CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

image_path = IMAGE_DIR / "000009.jpg"
label_path = LABEL_DIR / "000009.txt"

if not image_path.is_file():
    raise FileNotFoundError(f"Worked-example image not found: {image_path}")

if not label_path.is_file() or label_path.stat().st_size == 0:
    raise FileNotFoundError(f"Ground-truth annotation not found: {label_path}")

with Image.open(image_path) as image:
    image_width, image_height = image.size

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

ground_truth = []

for line_number, line in enumerate(
    label_path.read_text(encoding="utf-8").splitlines(), start=1
):
    values = line.split()

    if len(values) != 5:
        raise ValueError(f"Invalid annotation at line {line_number}.")

    try:
        class_value = float(values[0])
        xc, yc, width, height = map(float, values[1:])
    except ValueError as error:
        raise ValueError(f"Non-numeric annotation at line {line_number}.") from error

    if not all(math.isfinite(value) for value in [class_value, xc, yc, width, height]):
        raise ValueError(f"Non-finite annotation at line {line_number}.")

    if not class_value.is_integer():
        raise ValueError(f"Invalid class ID at line {line_number}.")

    class_id = int(class_value)

    if not (
        0 <= xc <= 1 and 0 <= yc <= 1
        and 0 < width <= 1 and 0 < height <= 1
        and xc - width / 2 >= 0 and yc - height / 2 >= 0
        and xc + width / 2 <= 1 and yc + height / 2 <= 1
    ):
        raise ValueError(f"Invalid bounding box at line {line_number}.")

    ground_truth.append((
        class_id,
        [
            (xc - width / 2) * image_width,
            (yc - height / 2) * image_height,
            (xc + width / 2) * image_width,
            (yc + height / 2) * image_height
        ]
    ))

if not ground_truth:
    raise ValueError("No ground-truth objects found in the worked-example image.")

prediction_result = best_model.predict(
    source=str(image_path),
    conf=CONFIDENCE_THRESHOLD,
    imgsz=640,
    device=0,
    verbose=False
)[0]

predictions = [
    (
        int(box.cls.item()),
        box.xyxy[0].cpu().tolist(),
        float(box.conf.item())
    )
    for box in prediction_result.boxes
]

matched_ground_truth = set()
true_positives = 0
false_positives = 0

for predicted_class, predicted_box, confidence in sorted(
    predictions, key=lambda item: item[2], reverse=True
):
    best_iou = 0.0
    best_match_index = None

    for index, (actual_class, actual_box) in enumerate(ground_truth):
        if index in matched_ground_truth or predicted_class != actual_class:
            continue

        iou = calculate_iou(predicted_box, actual_box)

        if iou > best_iou:
            best_iou = iou
            best_match_index = index

    if best_match_index is not None and best_iou >= IOU_THRESHOLD:
        true_positives += 1
        matched_ground_truth.add(best_match_index)
    else:
        false_positives += 1

false_negatives = len(ground_truth) - len(matched_ground_truth)

precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives else 0.0
)
recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives else 0.0
)
f1_score = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

print("=" * 50)
print("SINGLE TEST IMAGE EVALUATION")
print("=" * 50)
print("Image:", image_path.name)
print("Ground-truth objects:", len(ground_truth))
print("Predicted objects:", len(predictions))
print("True Positives (TP):", true_positives)
print("False Positives (FP):", false_positives)
print("False Negatives (FN):", false_negatives)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1_score:.4f}")
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("IoU matching threshold:", IOU_THRESHOLD)


### Previously Recorded Worked Example: 000009.jpg

The following results were recorded for the worked example using the Experiment 2 model, confidence threshold 0.25, and IoU threshold 0.50.

| Measure | Observed result |
|---|---:|
| Ground-truth objects | 4 |
| Predictions | 5 |
| True Positives (TP) | 4 |
| False Positives (FP) | 1 |
| False Negatives (FN) | 0 |
| Precision | 0.8000 (80.00%) |
| Recall | 1.0000 (100.00%) |
| F1-score | 0.8889 (88.89%) |

The four recorded matched detections were: person (confidence 0.7826, IoU 0.7437), horse (confidence 0.7457, IoU 0.8906), person (confidence 0.6043, IoU 0.6468), and person (confidence 0.5990, IoU 0.8462). All four matches exceeded the IoU threshold. One prediction was unmatched (FP), and no ground-truth objects were missed (FN = 0).

These are image-level results for one example, not the overall held-out test-set metrics. The code cell above recomputes the values when executed; if its output differs, use the newly computed output for the current dataset and checkpoint.
